# Parte 2: Limpieza de Datos y Curación

Tomando las decisiones extraídas del paso 1 (EDA), en este notebook aplicarás filtros definitivos para tu dataset.

## Objetivos
1. Crear funciones o pasos reproducibles para lidiar con valores nulos (borrado, imputación).
2. Lidiar con valores atípicos (*outliers*) de forma matemática o de negocio (ej. totales negativos o distancias nulas).
3. Separar las lógicas en celdas claras de forma que este código sea fácil de pasar a la carpeta `src/data/` y `src/features/`.
4. Exportar los datos intermedios listos para los pipelines.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from dotenv import load_dotenv
import snowflake.connector
load_dotenv('../.env')
conn = snowflake.connector.connect(
    user=os.getenv("SNOWFLAKE_USER"),
    password=os.getenv("SNOWFLAKE_PASSWORD"),
    account=os.getenv("SNOWFLAKE_ACCOUNT"),
    warehouse=os.getenv("SNOWFLAKE_WAREHOUSE"),
    database=os.getenv("SNOWFLAKE_DATABASE"),
    role=os.getenv("SNOWFLAKE_ROLE", "ACCOUNTADMIN")
)
# Filtramos directamente en SQL para traer SOLO los años de entrenamiento (2015-2023)
# Usamos la vista OBT_TRIPS_TRAIN que creamos previamente para mayor seguridad
query = "SELECT * FROM ANALYTICS.REFINED.OBT_TRIPS_TRAIN SAMPLE (1);"
print("Descargando muestra de entrenamiento (2015-2023) desde Snowflake...")
df = pd.read_sql(query, conn)
print(f"Carga finalizada. Registros: {len(df)}")
print(f"Años presentes en la muestra: {df['SOURCE_YEAR'].unique()}")
df.head()

Descargando muestra de entrenamiento (2015-2023) desde Snowflake...


C:\Users\Lhao\AppData\Local\Temp\ipykernel_35628\4087239922.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Carga finalizada. Registros: 7701137
Años presentes en la muestra: [2015 2016 2017 2018 2019 2020 2021 2022 2023]


,VENDOR_ID,PICKUP_DATETIME,DROPOFF_DATETIME,PASSENGER_COUNT,TRIP_DISTANCE,RATE_CODE_ID,STORE_AND_FWD_FLAG,PU_LOCATION_ID,DO_LOCATION_ID,PAYMENT_TYPE,...,EHAIL_FEE,TRIP_TYPE,SOURCE_YEAR,SOURCE_MONTH,SERVICE_TYPE,RUN_ID,INGESTED_AT_UTC,TRIP_DURATION_MIN,AVG_SPEED_MPH,TIP_PCT
0,1,2015-01-05 18:29:20,2015-01-05 18:32:00,1.0,0.60,1.0,N,229,170,1.0,...,NaN,NaN,2015,1,yellow,run_local_python_ingestion,2026-05-02 18:54:45.457589,2.67,13.50,28.75
1,2,2015-01-05 18:51:08,2015-01-05 19:05:24,2.0,3.07,1.0,N,236,41,1.0,...,NaN,NaN,2015,1,yellow,run_local_python_ingestion,2026-05-02 18:54:45.457589,14.27,12.91,27.04
2,1,2015-01-05 18:22:37,2015-01-05 18:29:17,1.0,1.20,1.0,N,68,68,2.0,...,NaN,NaN,2015,1,yellow,run_local_python_ingestion,2026-05-02 18:54:45.457589,6.67,10.80,0.00
3,1,2015-01-05 18:32:35,2015-01-05 18:39:05,1.0,1.20,1.0,N,140,263,1.0,...,NaN,NaN,2015,1,yellow,run_local_python_ingestion,2026-05-02 18:54:45.457589,6.50,11.08,31.85
4,2,2015-01-05 18:53:51,2015-01-05 19:00:10,1.0,0.96,1.0,N,249,113,2.0,...,NaN,NaN,2015,1,yellow,run_local_python_ingestion,2026-05-02 18:54:45.457589,6.32,9.12,0.00


In [14]:
print(len(df))

7701137


### 1. Filtros de Calidad de Datos

In [15]:
import pandas as pd
import numpy as np

# Reglas de limpieza refinadas tras el EDA:
df_clean = df.copy()

# 1. Filtro de Costo: Entre 1 USD y 500 USD (Elimina el outlier de $180k y viajes de $0)
df_clean = df_clean[(df_clean['TOTAL_AMOUNT'] >= 1.0) & (df_clean['TOTAL_AMOUNT'] <= 500.0)]

# 2. Filtro de Distancia: Entre 0.1 y 100 millas (Elimina errores de GPS y viajes estáticos)
df_clean = df_clean[(df_clean['TRIP_DISTANCE'] > 0.1) & (df_clean['TRIP_DISTANCE'] < 100.0)]

# 3. Filtro de Pasajeros: Al menos 1 pasajero y máximo 9
df_clean = df_clean[(df_clean['PASSENGER_COUNT'] >= 1) & (df_clean['PASSENGER_COUNT'] <= 9)]

# 4. Filtro de Duración: Mínimo 1 minuto, máximo 5 horas (Evita viajes instantáneos o fallos de reloj)
# Nota: La columna TRIP_DURATION_MIN ya la calculamos en SQL
df_clean = df_clean[(df_clean['TRIP_DURATION_MIN'] >= 1.0) & (df_clean['TRIP_DURATION_MIN'] <= 300.0)]

print(f"--- Resumen de Limpieza ---")
print(f"Filas originales: {len(df)}")
print(f"Filas limpias:    {len(df_clean)}")
print(f"Datos descartados: {len(df) - len(df_clean)} ({((len(df) - len(df_clean))/len(df)*100):.2f}%)")

# Verificamos que el outlier desapareció
print(f"\nValor máximo de TOTAL_AMOUNT ahora: ${df_clean['TOTAL_AMOUNT'].max()}")



--- Resumen de Limpieza ---
Filas originales: 7701137
Filas limpias:    7533634
Datos descartados: 167503 (2.18%)

Valor máximo de TOTAL_AMOUNT ahora: $497.46


### 2. Manejo de Nulos y Cardinalidad Categórica

In [17]:
# Rellenar nulos en variables de pago/cargos asumiendo 0
fill_zero_cols = ['CONGESTION_SURCHARGE', 'AIRPORT_FEE', 'EHAIL_FEE']
df_clean[fill_zero_cols] = df_clean[fill_zero_cols].fillna(0)

# Filtrar o imputar categóricas nulas
df_clean.dropna(subset=['PU_LOCATION_ID', 'DO_LOCATION_ID'], inplace=True)


In [18]:
print(len(df_clean))

7533634


### 3. Evitar Data Leakage

In [19]:
leakage_cols = [
    # Columnas de costo (suman el total)
    'FARE_AMOUNT', 'EXTRA', 'MTA_TAX', 'TIP_AMOUNT', 'TOLLS_AMOUNT',
    'IMPROVEMENT_SURCHARGE', 'CONGESTION_SURCHARGE', 'AIRPORT_FEE', 'EHAIL_FEE',
    'TIP_PCT',

    'PAYMENT_TYPE', 'TRIP_TYPE', 'STORE_AND_FWD_FLAG', 
    'RUN_ID', 'INGESTED_AT_UTC'
    
    # Columnas de "el futuro" (información post-viaje)
    'DROPOFF_DATETIME', 
    'TRIP_DURATION_MIN', 
    'AVG_SPEED_MPH'
]

df_clean.drop(columns=[col for col in leakage_cols if col in df_clean.columns], inplace=True)
print("Columnas actuales post-limpieza de leakage:", df_clean.columns)

Columnas actuales post-limpieza de leakage: Index(['VENDOR_ID', 'PICKUP_DATETIME', 'DROPOFF_DATETIME', 'PASSENGER_COUNT',
       'TRIP_DISTANCE', 'RATE_CODE_ID', 'PU_LOCATION_ID', 'DO_LOCATION_ID',
       'TOTAL_AMOUNT', 'SOURCE_YEAR', 'SOURCE_MONTH', 'SERVICE_TYPE',
       'INGESTED_AT_UTC'],
      dtype='object')


| Columna                  | Categoría    | Razón de la Eliminación                                                                          |
| ------------------------ | ------------ | ------------------------------------------------------------------------------------------------ |
| DROPOFF_DATETIME         | Data Leakage | Solo se conoce al finalizar el viaje. Incluirla permitiría calcular la duración real exacta.     |
| TRIP_DURATION_MIN        | Data Leakage | Es el factor determinante del precio. No está disponible en el momento de la predicción inicial. |
| FARE_AMOUNT              | Data Leakage | Es el componente principal de la variable objetivo (`TOTAL_AMOUNT`).                             |
| TIP_AMOUNT / TIP_PCT     | Data Leakage | La propina es discrecional y se decide una vez concluido el servicio.                            |
| EXTRA, MTA_TAX, etc.     | Data Leakage | Recargos (nocturnos, tráfico, etc.) que se calculan o confirman al final del trayecto.           |
| AVG_SPEED_MPH            | Data Leakage | Se deriva de la distancia y el tiempo real; información no disponible _a priori_.                |
| PAYMENT_TYPE             | Data Leakage | El método de pago se elige habitualmente al final. Sesga el reporte de propinas.                 |
| TRIP_TYPE                | Data Leakage | Clasificación técnica del servicio que puede variar durante el trayecto.                         |
| STORE_AND_FWD_FLAG       | Irrelevante  | Metadato técnico del sistema de grabación de datos GPS (memoria vs. transmisión).                |
| RUN_ID / INGESTED_AT_UTC | Metadato     | Identificadores únicos de la carga de datos que no tienen valor predictivo.                      |

### 4. Guardar Dataset Intermedio (Opcional)

In [20]:
# Definimos la query de limpieza masiva
sql_clean_view = """
CREATE OR REPLACE VIEW ANALYTICS.REFINED.OBT_TRIPS_CLEAN AS
SELECT * 
FROM ANALYTICS.REFINED.OBT_TRIPS
WHERE TOTAL_AMOUNT BETWEEN 1.0 AND 500.0
  AND TRIP_DISTANCE BETWEEN 0.1 AND 100.0
  AND PASSENGER_COUNT BETWEEN 1 AND 9
  AND TRIP_DURATION_MIN BETWEEN 1.0 AND 300.0
  AND SOURCE_YEAR <= 2023; -- Aseguramos que la limpieza base solo afecte a entrenamiento
"""

print("Enviando lógica de limpieza a Snowflake...")
conn.cursor().execute(sql_clean_view)
print("¡Éxito! Ahora tienes una vista limpia en la nube sin haber gastado RAM local.")


Enviando lógica de limpieza a Snowflake...
¡Éxito! Ahora tienes una vista limpia en la nube sin haber gastado RAM local.
